# VQCC v14: PSD chemistry and projective DQI

Full local original-verifier score **22.90331826**, previous v13 **22.81748526**.
Our exact-moment/norm PSD continuation lowers chemistry lambda_eff to
48.4764209747 with unchanged public ANGLE plans/words. Kirill's comparison
identified hengck23's cheaper889Q projective DQI; that binary is used unchanged.
Atakan's46-word chemistry/portable emitter and the full upstream credit chain
are retained. Busyaprime's63Q inverse-square-root circuit is unchanged.
The DQI is approximate: known rare-weight failures remain disclosed and all
observed errors enter the full official repetition cost. Local score is not LB.
CPU coefficient reproduction,15 byte-exact loader rebuilds, every-address
both-direction/all-HMR proofs and a missing-CZ negative control are included.


## The host's target-register contract

[Tanuj Khattar's official clarification](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092) requires correctness on the
states reached in the specified forward and reverse traversals. Forward
ANGLE_k starts from batch k-1 (zero for k=0); reverse starts from batch k.
The current public loaders use the sign encoding bit7=bit6 on those states.

Our previous interpretation unnecessarily required arbitrary incoming words.
That stronger test does fail on these public loaders; this does not invalidate
them under the host's ruling. We explicitly make no arbitrary-target claim.
The algebraic audit below checks all 1220 addresses, both required starting
states, clean workspace, restored address and all 2^80 HMR outcomes per circuit.


In [ ]:
import glob, hashlib, io, json, lzma, math, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import numpy as np

INPUT=Path(os.environ.get('QCC_INPUT_DIR','/kaggle/input'))
WORK=Path(os.environ.get('QCC_WORK_DIR','/kaggle/working'));WORK.mkdir(parents=True,exist_ok=True)
def sha(path):
    with Path(path).open('rb')as f:return hashlib.file_digest(f,'sha256').hexdigest()
matches=sorted(Path(p)for p in glob.glob(str(INPUT/'**/components_manifest_v14.json'),recursive=True))
assert len(matches)==1,'Attach the pinned v14 components dataset'
mp=matches[0];assert sha(mp)=='04bd5903d030ccf2b0424edb91a5432f2b39f6062a7ab7ae6bb48f6e1f1f98ee'
MANIFEST=json.loads(mp.read_text());DATA=mp.parent
options=[Path(os.environ.get('QCC_COMP_DIR','/nonexistent')),
 INPUT/'competitions/google-verifiable-quantum-circuit-compilation',
 INPUT/'google-verifiable-quantum-circuit-compilation']
COMP=next((p for p in options if(p/'kickmix').is_dir()and(p/'verifiers').is_dir()),None)
assert COMP is not None,'Attach the competition data'
env=dict(os.environ,QCC_COMP_DIR=str(COMP),OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
env['PYTHONPATH']=os.pathsep.join([str(COMP/'kickmix'),str(COMP/'verifiers'),str(COMP/'verifiers/dqi')])
SUB=WORK/'submission_v14';SUB.mkdir(exist_ok=True)
for item in MANIFEST['files']:
    src=DATA/item['stored_as'];assert sha(src)==item['stored_sha256']
    dst=SUB/item['file'];dst.parent.mkdir(parents=True,exist_ok=True)
    if item['xz']:
        with lzma.open(src,'rb')as fi,dst.open('wb')as fo:shutil.copyfileobj(fi,fo)
    else:shutil.copyfile(src,dst)
    assert sha(dst)==item['sha256']
bundle=DATA/MANIFEST['source_bundle']['stored_as'];assert sha(bundle)==MANIFEST['source_bundle']['sha256']
SOURCE=WORK/'v14_source';SOURCE.mkdir(exist_ok=True)
with lzma.open(bundle,'rb')as f,zipfile.ZipFile(io.BytesIO(f.read()))as z:z.extractall(SOURCE)
for name,pin in json.loads((SOURCE/'source_manifest.json').read_text()).items():assert sha(SOURCE/name)==pin,name
print('Authenticated 18 submission files and whitelisted reproduction sources.')


## Reproduce the selected coefficient update

The pinned public Atakan factor plus our selected delta reproduces W/H2_I and derived offset_W on CPU. Six other arrays are byte-identical to that public factor.


In [ ]:
reproduced=WORK/'reproduced_coefficients.npz'
subprocess.run([sys.executable,str(SOURCE/'our/apply_weight_delta.py'),
 str(SOURCE/'public/atakan_original.npz'),str(SOURCE/'our/selected_weight_delta.npz'),str(reproduced)],env=env,check=True)
with np.load(reproduced,allow_pickle=False)as got,np.load(SUB/'dfthcsos/submission.npz',allow_pickle=False)as expected:
    assert set(got.files)==set(expected.files)
    for key in got.files:
        if key in ('W','H2_I','offset_W'):assert np.allclose(got[key],expected[key],rtol=0,atol=1e-11),key
        else:assert got[key].dtype==expected[key].dtype and got[key].tobytes()==expected[key].tobytes(),key
    for key,pin in MANIFEST['immutable_arrays'].items():assert hashlib.sha256(expected[key].tobytes()).hexdigest()==pin,key
print('Coefficient reproduction and all six immutable-array checks PASS.')


In [ ]:
import tempfile
rebuilt=Path(tempfile.mkdtemp(prefix='v14-native-',dir=WORK))/'build'
subprocess.run([sys.executable,str(SOURCE/'public_chemistry/rebuild_native.py'),'--out',str(rebuilt)],env=env,check=True)
for item in MANIFEST['files']:
    if item['file'].startswith('dfthcsos/')and item['file'].endswith('.kmb'):
        assert sha(rebuilt/item['file'])==item['sha256'],item['file']
print('All15 production ANGLE files rebuilt byte-exactly from credited public plans.')


## Exact reachable-state phase check

The bit-packed evaluator is adapted from mrmeows' public audit. Each address
is evaluated exactly; every HMR has its own symbolic phase coefficient. All
coefficients must be zero. The proof checks actual circuit text. A missing-CZ
mutation must fail as a negative control. This is an independent encoder of
the allowed gate semantics, supplemented by the official native simulation.


In [ ]:
proof_path=WORK/'reachable_proof.json'
subprocess.run([sys.executable,str(SOURCE/'our/verify_reachable.py'),str(SUB/'dfthcsos'),str(proof_path)],env=env,check=True)
proof=json.loads(proof_path.read_text());assert proof['status']=='PASS' and len(proof['circuits'])==15
subprocess.run([sys.executable,str(SOURCE/'our/proof_negative_control.py'),str(SUB/'dfthcsos')],env=env,check=True)


## Run the three complete official verifiers

Fresh isolated subprocesses, unchanged verifier files and full default shot
counts: 200,540,928 inverse-square-root shots, 102,400 DQI shots and 78,080
chemistry shots. The declared salt is notebook-verification. DQI failures,
if observed, remain in its repetition penalty and receipt; no salt retries.


In [ ]:
for task,filename in [('invsqrt','invsqrt/verify_submission_invsqrt.py'),('dqi','dqi/verify_submission_dqi.py'),('dfthcsos','dfthcsos/verify.py')]:
    assert sha(COMP/'verifiers'/filename)==MANIFEST['verifier_sha256'][task],task
results={};started=time.time()
for task in ('invsqrt','dqi','dfthcsos'):
    path=SUB/('dfthcsos'if task=='dfthcsos'else task+'.kmb')
    report=WORK/(task+'_official.json')
    subprocess.run([sys.executable,str(SOURCE/'our/verify_task.py'),task,str(path),str(report),'--workers','4'],env=env,check=True)
    result=json.loads(report.read_text());assert result['status']=='PASS';results[task]=result
ratios={task:MANIFEST['reference_costs'][task]/r['score']for task,r in results.items()}
gm=math.prod(ratios.values())**(1/3)
print('Fresh official results:',json.dumps({t:{'status':r['status'],'cost':r['score'],'Q':r['num_qubits']}for t,r in results.items()}))
print(f'Local geometric mean: {gm:.8f}; elapsed {time.time()-started:.1f}s')
print('Control Alex v54:',MANIFEST['alex_v54_control']['geometric_mean'],'; our previous v12:',MANIFEST['previous_our_v13'])
assert gm>MANIFEST['previous_our_v13'],'Candidate did not improve on the pinned v12 result'


## Export the verified files

The submission contains exactly the 18 authenticated production members.
Every compressed member is read back and hashed. The verification receipt
and reachable-state certificate are separate outputs. Running this notebook
does not make a leaderboard submission.


In [ ]:
archive=WORK/'submission.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6)as z:
    for item in MANIFEST['files']:z.write(SUB/item['file'],item['file'])
with zipfile.ZipFile(archive)as z:
    assert len(z.namelist())==18 and set(z.namelist())=={i['file']for i in MANIFEST['files']}
    assert z.testzip()is None
    for item in MANIFEST['files']:
        with z.open(item['file'])as f:assert hashlib.file_digest(f,'sha256').hexdigest()==item['sha256']
receipt=dict(release=MANIFEST['release'],salt=MANIFEST['salt'],local_geometric_mean=gm,
 results=results,ratios=ratios,archive_sha256=sha(archive),archive_bytes=archive.stat().st_size,
 files=MANIFEST['files'],reachable_proof='reachable_proof.json',leaderboard_submission=False)
(WORK/'verification_v14.json').write_text(json.dumps(receipt,indent=2))
print('Wrote verified 18-member submission.zip; SHA-256',sha(archive))
print('All work complete. This is a local verifier score, not a new leaderboard claim.')


## Optional PSD search

The source README includes portable CUDA commands for the PSD proxy and exact-moment/norm continuation. The notebook uses the deterministic CPU coefficient reproduction and runs the selected pinned files through the original verifiers. Optimization reruns can converge differently.


## Sources and attribution

# Sources and attribution

- **Selected chemistry update:** Atakan Aldemir's public native chemistry
  co-design bank, downloaded from the completed notebook output:
  https://www.kaggle.com/code/atakanaldemir/qcc-888q-repair-and-native-chemistry-co-design
  Its compiler-constrained46-word update, physical/LP-refitted coefficients,
  recovered plans and portable native emitter are Atakan's contributions.
  This release inherits that selected chemistry and credits it explicitly.
  The three ANDs saved per pass are credited to **Heliosli (two)** and
  **Dmitrii Gluzdov (one)**, as recorded in the upstream bank.
  https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy
  https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source
- **Our selected chemistry change:** Ahmed Berat Ozer, augmented-PSD
  relaxation/rank projection followed by exact moment/global-offset-norm
  factor continuation and actual augmented-L1 optimization. The selected
  coefficients lower lambda_eff from our v13's48.5596155805 to48.4764209747
  with the same quantized angles,15 public circuits and all physical checks.
  Convex entrywise-Gram regularization provides a warm start; it is not an
  optimality certificate. Portable optional CUDA code and numerical records
  are included. CPU deltas reproduce the pinned selected arrays. The PSD,
  proximal and manifold methods have established literature; this is our
  implementation and application, not invention of those general methods.
- **Chemistry ancestry:** SomeJoint/Agniv Banerjee's R8/B143/C60 factor family
  and paired-product compiler, Michael Rustad, Ahmed Berat Ozer, busyaprime,
  lightningv08/Vladislav Lavrentev, mrmeows and Dmitrii Gluzdov. Exact-gauge
  and coefficient optimization prior art includes Kirill Tymchishin,
  mrmeows, Kozykorpesh, Alex Chilton and Phuc Thai. The public bank's
  CREDITS.md, LICENSE.md, SOURCE_MANIFEST.json and inherited notices are
  retained with their original scope and historical labels.
  https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2
  https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom
- **Selected DQI:** hengck23's unchanged public889Q projective decoder,
  iteration014 / notebookV1 run363497772: projective extraction/reverse,
  rolling continuants, tail/head folds and streamed-converter integration.
  https://www.kaggle.com/code/hengck23/dqi-kmb-at-59-56854?scriptVersionId=363497772
  https://www.kaggle.com/datasets/hengck23/hengck23-dqi-59-57
  It deliberately approximates rare weight branches. Observed official
  failures are charged in the full repetition penalty. No exact decoder
  claim or source-generator rebuild is made; only its public binary is used.
- **Kirill Tymchishin:** independent full decoder comparisons, exact-byte
  portfolio integration and MPS reachable-phase audit/mutation test, which
  led us to the selected hengck23 binary. Those are Kirill's contributions.
  https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom
  Kirill's attribution/limitations file is retained verbatim below sources.
- **Decoder ancestry and prior control:** SomeJoint/Agniv Banerjee's889Q
  paired charts, inverse19 reverse, streamed divide and head schedule,
  previously selected in our v12/v13. The chain also credits Ahmed Ozer's
  earlier narrowing, Alex Chilton's Schur design on jday96314/QuantumAI,
  mrmeows/Ozer/manwithacat mask projection, Mudit Chaturvedi, Dmitrii Gluzdov,
  ViberTeam and Phuc Thai. The original jday repository has no explicit
  licence; its source is not redistributed here and no blanket licence
  is asserted over that history.
- **Inverse square root:** busyaprime/Akmal Xodarev's updated63Q file,
  unchanged from our v12 and Alex Chilton's public portfolio. It rebuilds
  two regions of mrmeows'63Q circuit; lineage also includes Phuc Thai/
  DePie, Dmitrii Gluzdov and marsyas. Apache-2.0 as supplied upstream.
  https://www.kaggle.com/code/busyaprime/lb-22-72431-22x-cheaper-than-google-s-circuits
  https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio
- **Exact reachable phase proof:** our adaptation of mrmeows' bit-packed
  audit; original Apache-2.0 notice is retained. The native production
  circuit text is checked at all1220addresses and both traversal inputs,
  for every independent HMR outcome. No arbitrary-word claim is made.
  https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits
- **Official contract:** Tanuj Khattar's host clarification:
  https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092
- **Organizers:** Google Quantum AI / Qualtran; unchanged official verifiers,
  task definitions, fixed-shift Hamiltonian and Kickmix runtime.

Our scripts are Apache-2.0. Every upstream asset retains its own terms;
historical licence text does not relicense unrelated third-party sources.
